In [ ]:
!pip install torch peft transformers numpy accelerate trl datasets --quiet
!pip install -U bitsandbytes --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 38.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import copy, torch, numpy
from transformers import AutoModelForCausalLM
from accelerate import init_empty_weights, load_checkpoint_and_dispatch
from peft import PeftModel

In [ ]:
finetuned_model = "/content/drive/MyDrive/CAISI_phase2/qlora-finetuning/16bit-checkpoints/"

class SafeLoRAConfig:
  base_model_path = "meta-llama/Meta-Llama-3.1-8B"
  aligned_model_path = "meta-llama/Meta-Llama-3.1-8B-Instruct"
  devices = "cuda"
  select_layers_type = "threshold"
  threshold = 0.5
  num_proj_layers = 5


In [ ]:
config = SafeLoRAConfig()

In [ ]:
print(config.aligned_model_path, config.base_model_path)

meta-llama/Meta-Llama-3.1-8B-Instruct meta-llama/Meta-Llama-3.1-8B


In [ ]:
# kp attempt
# model = AutoModelForCausalLM.from_pretrained(config.aligned_model_path)

In [ ]:
pmodel = AutoModelForCausalLM.from_pretrained(finetuned_model,dtype=torch.float16)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
pmodel.config

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "float16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "transformers_version": "4.57.1",
  "use_cache": true,
  "vocab_size": 128256
}

In [ ]:
# class SafeLoRAConfig:
#   base_model_path = "meta-llama/Meta-Llama-3.1-8B"
#   aligned_model_path = base_model_path

# Copyright 2023-present the HuggingFace Inc. team.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.



class SafeLoRA:
    def __init__(self, peft_model:torch.nn.Module, config):
        """
        Please use safelora.model to get the projected model.

        How to use SafeLoRA:
        path = './LLM_Models/llama-2-7b-chat-fp16/' # load your base model of the peft model
        model = AutoModelForCahowusalLM.from_pretrained(path)
        pmodel = PeftModel.from_pretrained(model, 'finetuneLLM/finetuned_models/samsumBad-7b-fp16-peft-seed-42/',torch_dtype=torch.float16) #load peft model

        SafeLoRAConfig.base_model_path = './LLM_Models/llama-2-7b-hf/'  #you should modify the path
        SafeLoRAConfig.aligned_model_path = './LLM_Models/llama-2-7b-chat-fp16/' #you should modify the path

        safelora = SafeLoRA(pmodel, SafeLoRAConfig)

        Finally, you can get the projected model by "safelora.model".
        """
        super().__init__()
        self.peft_model = peft_model
        self.config = config
        self.peft_config = peft_model.peft_config["default"]
        self.model_ori = copy.deepcopy(peft_model)
        print("Getting projection matrix:")
        project_matrix = self.get_aligned_matrix()
        if self.config.select_layers_type == 'threshold':
            print("Selecting layers by threshold:")
            self.model, _ = self.projected_weighted(project_matrix, self.config.threshold, show_info=True)
        elif self.config.select_layers_type == 'number':
            model, cos = self.projected_weighted(project_matrix, -1, show_info=False)
            thrs = numpy.sort(cos)[:self.config.num_proj_layers][-1]
            self.model, _ = self.projected_weighted(project_matrix, thrs, show_info=True)
        else:
            raise ValueError("The method of select_layer_type should be threshold or number.")

    def get_aligned_matrix(self):
        """
        Get projected matrix by following the config (target_modules) from the peft model.
        The dimensions between the base model's weights and the aligned model's weights should be the same.
        """
        print("Loading base model: {}".format(self.config.base_model_path))
        base_model = AutoModelForCausalLM.from_pretrained(
            self.config.base_model_path,
            return_dict=True,
            load_in_8bit=False,
            device_map="cpu",
            low_cpu_mem_usage=True,
        )

        print("Loading aligned model: {}".format(self.config.aligned_model_path))
        aligned_model = AutoModelForCausalLM.from_pretrained(
            self.config.aligned_model_path,
            return_dict=True,
            load_in_8bit=False,
            device_map="cpu",
            low_cpu_mem_usage=True,
        )
        v = []
        proj_modules = list(self.peft_config.target_modules)
        for (b_name, b_param) , (a_name, a_param) in zip (base_model.named_parameters(), aligned_model.named_parameters()):
            if any(module in a_name for module in proj_modules):
                print("Layer {}".format(a_name, b_name))
                assert b_param.shape == a_param.shape, "The dimensions of the base model's weight should be the same with the aligned model's weight."
                vec = a_param - b_param
                vec = vec.to(self.config.devices)
                vec = torch.mm(vec, vec.t()) / torch.norm(vec)
                v.append((vec).detach().cpu())
        return v

    def projected_weighted(self, project_matrix, thrs_cos, show_info=False):
        v = project_matrix
        idx = 0
        i = 0
        dis = []
        cos_total = []
        for (name, param),(name_ori, param_ori) in zip(self.peft_model.named_parameters(), self.model_ori.named_parameters()):
            if 'lora' in name:
                if param.shape[0] == self.peft_config.r:
                    B = copy.deepcopy(param_ori)
                if param.shape[0] != self.peft_config.r:
                    P = v[idx].to(device=param.device, dtype=param.dtype)
                    W = torch.mm(P, param_ori.data)
                    fW = torch.mm(W, B)
                    ori = torch.mm(param_ori, B)
                    W_new = torch.mm(P, param_ori.data)
                    cos = numpy.round(torch.nn.functional.cosine_similarity(fW.reshape(1,-1), ori.reshape(1,-1)).item(),5)
                    cos_total.append(cos)

                    if cos <=  thrs_cos:
                        i+=1
                        param.data =  W_new
                    else:
                        param.data = param_ori
                    dist = 1 / (1+torch.norm(param.data.reshape(1,-1)-W.reshape(1,-1)))

                    dis.append(dist.item())
                    idx += 1
        if show_info:
            print(f"{i} layers are projected, cosine threshold is {thrs_cos}, and Pdst is {numpy.mean(dis)} (> 0.8 is better).")
        return self.peft_model, cos_total


In [ ]:
safeLora = SafeLoRA(pmodel, SafeLoRAConfig)

Getting projection matrix:
Loading base model: meta-llama/Meta-Llama-3.1-8B


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Loading aligned model: meta-llama/Meta-Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Layer model.layers.0.self_attn.q_proj.weight
Layer model.layers.0.self_attn.v_proj.weight
Layer model.layers.1.self_attn.q_proj.weight
Layer model.layers.1.self_attn.v_proj.weight
Layer model.layers.2.self_attn.q_proj.weight
Layer model.layers.2.self_attn.v_proj.weight
Layer model.layers.3.self_attn.q_proj.weight
Layer model.layers.3.self_attn.v_proj.weight
Layer model.layers.4.self_attn.q_proj.weight
Layer model.layers.4.self_attn.v_proj.weight
Layer model.layers.5.self_attn.q_proj.weight
Layer model.layers.5.self_attn.v_proj.weight
Layer model.layers.6.self_attn.q_proj.weight
Layer model.layers.6.self_attn.v_proj.weight
Layer model.layers.7.self_attn.q_proj.weight
Layer model.layers.7.self_attn.v_proj.weight
Layer model.layers.8.self_attn.q_proj.weight
Layer model.layers.8.self_attn.v_proj.weight
Layer model.layers.9.self_attn.q_proj.weight
Layer model.layers.9.self_attn.v_proj.weight
Layer model.layers.10.self_attn.q_proj.weight
Layer model.layers.10.self_attn.v_proj.weight
Layer mo

In [ ]:
projected_model = safeLora.model
projected_model = projected_model.to("cuda")
projected_model.save_pretrained("/content/drive/MyDrive/CAISI_phase2/SafeLoRA-finetuning/saved_weights")